# Row Level Security (RLS) — `vstone_catalog.security`

| Group | Brands visible |
|---|---|
| `admin_group` | All brands (unrestricted) |
| `premium_users` | `BMW`, `Mercedes-Benz`, `Lexus` |
| `toyota_users` | `Toyota` only |
| `honda_users` | `Honda` only |
| *(others)* | No rows (deny by default) |

## Step 1 — Inspect base Gold table

In [0]:
SELECT * FROM vstone_catalog.gold.agg_top_10_brands_by_spend
ORDER BY total_market_value_usd DESC;

## Step 2 — Verify workspace groups

In [0]:
SHOW GROUPS;

## Step 3 — Confirm user identity and group membership

In [0]:
SELECT
  CURRENT_USER()                  AS current_user,
  IS_MEMBER('toyota_users')       AS is_toyota_user,
  IS_MEMBER('honda_users')        AS is_honda_user,
  IS_MEMBER('premium_users')      AS is_premium_user,
  IS_MEMBER('admin_group')        AS is_admin;

## Step 4 — Create RLS view on `agg_top_10_brands_by_spend`

In [0]:
-- ============================================================
-- RLS VIEW: agg_top_10_brands_by_spend
-- IS_MEMBER() used -- works with workspace-local groups (Free Edition).
-- LOWER(TRIM(brand)) handles mixed case: dim_car stores 'Toyota', 'BMW'
-- (clean_text UDF strips only, preserves case).
-- ============================================================
CREATE OR REPLACE VIEW vstone_catalog.security.rls_brand_market_data AS
SELECT
  brand,
  total_market_value_usd,
  total_listings,
  avg_price_usd,
  gold_load_dt
FROM vstone_catalog.gold.agg_top_10_brands_by_spend
WHERE
  CASE
    WHEN IS_MEMBER('admin_group')    THEN TRUE
    WHEN IS_MEMBER('premium_users')  THEN LOWER(TRIM(brand)) IN ('bmw', 'mercedes-benz', 'lexus')
    WHEN IS_MEMBER('toyota_users')   THEN LOWER(TRIM(brand)) = 'toyota'
    WHEN IS_MEMBER('honda_users')    THEN LOWER(TRIM(brand)) = 'honda'
    ELSE FALSE
  END;

## Step 5 — Verify filtered output

In [0]:
SELECT * FROM vstone_catalog.security.rls_brand_market_data
ORDER BY total_market_value_usd DESC;

## Step 6 — Create RLS view on `fact_listings`

In [0]:
-- ============================================================
-- RLS VIEW: fact_listings
-- fact_listings has no brand/model/fuel_type/price_category columns.
-- Resolved via dim joins. RLS WHERE applied on dc.brand.
-- ============================================================
CREATE OR REPLACE VIEW vstone_catalog.security.rls_fact_listings AS
SELECT
  f.listing_id,
  dc.brand,
  dc.model,
  f.manufacture_year,
  f.listing_date,
  f.price_rub,
  f.price_usd,
  dp.price_category,
  f.mileage_km,
  dc.fuel_type,
  dl.city_name,
  f.photo_count,
  f.car_age_at_listing,
  f.is_high_mileage,
  f.price_per_hp_usd,
  f.gold_load_dt
FROM vstone_catalog.gold.fact_listings f
LEFT JOIN vstone_catalog.gold.dim_car            dc ON f.car_sk             = dc.car_sk
                                                   AND dc.__END_AT IS NULL
LEFT JOIN vstone_catalog.gold.dim_price_category dp ON f.price_category_key = dp.price_category_key
LEFT JOIN vstone_catalog.gold.dim_location       dl ON f.location_sk        = dl.location_sk
                                                   AND dl.__END_AT IS NULL
WHERE
  CASE
    WHEN IS_MEMBER('admin_group')   THEN TRUE
    WHEN IS_MEMBER('premium_users') THEN LOWER(TRIM(dc.brand)) IN ('bmw', 'mercedes-benz', 'lexus')
    WHEN IS_MEMBER('toyota_users')  THEN LOWER(TRIM(dc.brand)) = 'toyota'
    WHEN IS_MEMBER('honda_users')   THEN LOWER(TRIM(dc.brand)) = 'honda'
    ELSE FALSE
  END;

-- Verify
SELECT brand, COUNT(*) AS row_count
FROM vstone_catalog.security.rls_fact_listings
GROUP BY brand
ORDER BY row_count DESC;